# 05 - Data Enrichment - Calas x Oceanographic Matching

Matches each daily fishing event to the nearest HYCOM and MODIS grid point in space and time to attach environmental variables.

**Steps:**
1. Load the daily calas table produced by notebook 03
2. For each fishing event, find the nearest HYCOM cell and extract water temperature and salinity
3. Attach HYCOM temperature anomaly (from notebook 01 outputs)
4. Attach MODIS chlorophyll-a (from notebook 04 outputs)
5. Attach MODIS SST and SST anomaly (from notebook 04b outputs)
6. Compute salinity anomaly relative to 35.1 PSU reference
7. Save the enriched dataset: `calas_with_hycom_data.csv`

In [2]:
import pandas as pd
import xarray as xr
import numpy as np
from pathlib import Path
from scipy.spatial.distance import cdist
from pyproj import datadir
datadir.set_data_dir("/home/jupyter-daniela/.conda/envs/peru_environment/share/proj")


In [ ]:
calas_path = Path("/home/jupyter-daniela/suyana/peru_production/outputs/")
hycom_path = Path("/home/jupyter-daniela/suyana/peru_production/features/")
output_path = Path("/home/jupyter-daniela/suyana/peru_production/outputs/")

df_calas = pd.read_csv(calas_path / "calas_daily_with_coordinates.csv")
df_calas['date'] = pd.to_datetime(df_calas['date'])
df_calas['year'] = df_calas['date'].dt.year


In [ ]:
def match_calas_vectorized(df, ds, var):
    """Vectorized nearest-neighbor match of df lat/lon/date against an xarray dataset."""
    dates = xr.DataArray(df['date'].values, dims='points')
    lats  = xr.DataArray(df['lat'].values,  dims='points')
    lons  = xr.DataArray(df['lon'].values,  dims='points')
    vals  = ds[var].sel(time=dates, lat=lats, lon=lons, method='nearest').values
    return vals.astype(float)

In [ ]:
years = sorted(df_calas['year'].unique())
all_results = []

for year in years:
    df_year = df_calas[df_calas['year'] == year].copy()

    temp_file = hycom_path / f"hycom_water_temp_daily_{year}.nc"
    salt_file = hycom_path / f"hycom_salinity_daily_{year}.nc"
    if not temp_file.exists() or not salt_file.exists():
        print(f"{year}: no HYCOM, skipping")
        continue

    print(f"{year}: HYCOM temp/sal...", end=' ', flush=True)
    ds_temp = xr.open_dataset(temp_file)
    ds_salt = xr.open_dataset(salt_file)
    df_year['hycom_temp'] = match_calas_vectorized(df_year, ds_temp, 'water_temp')
    df_year['hycom_sal']  = match_calas_vectorized(df_year, ds_salt, 'salinity')
    ds_temp.close(); ds_salt.close()
    print("done")

    anom_file = hycom_path / f"hycom_water_temp_anomaly_daily_{year}.nc"
    if anom_file.exists():
        print(f"{year}: HYCOM temp anom...", end=' ', flush=True)
        ds = xr.open_dataset(anom_file)
        df_year['hycom_temp_anom'] = match_calas_vectorized(df_year, ds, 'water_temp_anomaly')
        ds.close()
        print("done")
    else:
        df_year['hycom_temp_anom'] = np.nan

    chl_file = hycom_path / f"chl_daily_{year}.nc"
    if chl_file.exists():
        print(f"{year}: chl...", end=' ', flush=True)
        ds = xr.open_dataset(chl_file)
        df_year['chl'] = match_calas_vectorized(df_year, ds, 'chlor_a')
        ds.close()
        print("done")
    else:
        df_year['chl'] = np.nan

    sst_file = hycom_path / f"sst_daily_{year}.nc"
    if sst_file.exists():
        print(f"{year}: modis sst...", end=' ', flush=True)
        ds = xr.open_dataset(sst_file)
        df_year['modis_sst'] = match_calas_vectorized(df_year, ds, 'sst')
        ds.close()
        print("done")
    else:
        df_year['modis_sst'] = np.nan

    sst_anom_file = hycom_path / f"sst_anomaly_daily_{year}.nc"
    if sst_anom_file.exists():
        print(f"{year}: modis sst anom...", end=' ', flush=True)
        ds = xr.open_dataset(sst_anom_file)
        df_year['modis_sst_anom'] = match_calas_vectorized(df_year, ds, 'sst_anomaly')
        ds.close()
        print("done")
    else:
        df_year['modis_sst_anom'] = np.nan

    all_results.append(df_year)
    print(f"{year}: COMPLETE ({len(df_year):,} rows)\n")

df_final = pd.concat(all_results, ignore_index=True)
print(f"Total: {len(df_final):,} rows")

In [ ]:
anom_vals = []
for year in sorted(df_final['date'].dt.year.unique()):
    df_year = df_final[df_final['date'].dt.year == year]
    anom_file = hycom_path / f"hycom_water_temp_anomaly_daily_{year}.nc"
    if not anom_file.exists():
        anom_vals.extend([np.nan] * len(df_year))
        continue
    print(f"{year}: temp anomaly...", end=' ')
    ds = xr.open_dataset(anom_file)
    anom_vals.extend(match_calas_vectorized(df_year, ds, 'water_temp_anomaly'))
    ds.close()
    print("done")

df_final['hycom_temp_anom'] = anom_vals

In [ ]:
chl_vals = []
chl_anom_vals = []
for year in sorted(df_final['date'].dt.year.unique()):
    df_year = df_final[df_final['date'].dt.year == year]

    chl_file = hycom_path / f"chl_daily_{year}.nc"
    if chl_file.exists():
        print(f"{year}: chl...", end=' ', flush=True)
        ds = xr.open_dataset(chl_file)
        chl_vals.extend(match_calas_vectorized(df_year, ds, 'chlor_a'))
        ds.close()
        print("done")
    else:
        chl_vals.extend([np.nan] * len(df_year))

    anom_file = hycom_path / f"chl_anomaly_daily_{year}.nc"
    if anom_file.exists():
        print(f"{year}: chl anom...", end=' ', flush=True)
        ds = xr.open_dataset(anom_file)
        chl_anom_vals.extend(match_calas_vectorized(df_year, ds, 'chl_anomaly'))
        ds.close()
        print("done")
    else:
        chl_anom_vals.extend([np.nan] * len(df_year))

df_final['chl'] = chl_vals
df_final['chl_anom'] = chl_anom_vals

In [ ]:
sst_vals = []
for year in sorted(df_final['date'].dt.year.unique()):
    df_year = df_final[df_final['date'].dt.year == year]
    sst_file = hycom_path / f"sst_daily_{year}.nc"
    if not sst_file.exists():
        sst_vals.extend([np.nan] * len(df_year))
        continue
    print(f"{year}: modis sst...", end=' ')
    ds = xr.open_dataset(sst_file)
    sst_vals.extend(match_calas_vectorized(df_year, ds, 'sst'))
    ds.close()
    print("done")

df_final['modis_sst'] = sst_vals

In [ ]:
sst_anom_vals = []
for year in sorted(df_final['date'].dt.year.unique()):
    df_year = df_final[df_final['date'].dt.year == year]
    anom_file = hycom_path / f"sst_anomaly_daily_{year}.nc"
    if not anom_file.exists():
        sst_anom_vals.extend([np.nan] * len(df_year))
        continue
    print(f"{year}: modis sst anomaly...", end=' ')
    ds = xr.open_dataset(anom_file)
    sst_anom_vals.extend(match_calas_vectorized(df_year, ds, 'sst_anomaly'))
    ds.close()
    print("done")

df_final['modis_sst_anom'] = sst_anom_vals

In [ ]:
df_final['hycom_sal_anom_ref35'] = df_final['hycom_sal'] - 35.1


In [13]:
df_final.to_csv(output_path / "calas_with_hycom_data.csv", index=False)